In [3]:
# %%
# =============================================================================
# 02_rag_pipeline.ipynb
# Financial AI Governance — RAG Pipeline (ChromaDB)
# Kernel : Python (llm_env)
# Input  : data/regulations/*.md (3 regulation documents)
#          data/processed/dataset_final.json
# Output : vectordb/nist/        (Chroma persistent store)
#          vectordb/eu_aiact/     (Chroma persistent store)
#          vectordb/kr_aibasicact/(Chroma persistent store)
#          results/tables/table3_rag_config.csv
# =============================================================================

# %%
# =============================================================================
# Cell 1. Libraries and Environment Setup
# =============================================================================
import os
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from dotenv import load_dotenv

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# Directory paths
REG_DIR   = '../data/regulations'
VDB_DIR   = '../vectordb'
TABLE_DIR = '../results/tables'
DATA_DIR  = '../data/processed'

for d in [TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

# API setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

# Embedding model
EMBED_MODEL = 'text-embedding-3-small'
embeddings  = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=OPENAI_API_KEY
)

print("[INFO] Environment setup complete")
print(f"  Embedding model : {EMBED_MODEL}")
print(f"  REG_DIR         : {os.path.abspath(REG_DIR)}")
print(f"  VDB_DIR         : {os.path.abspath(VDB_DIR)}")


# %%
# =============================================================================
# Cell 2. Regulation Document Configuration
# =============================================================================
# Each regulation: md file, chroma persist dir, chunking header, collection name
REGULATIONS = [
    {
        'name'       : 'NIST AI RMF',
        'key'        : 'nist',
        'file'       : 'nist_rmf_md.md',
        'persist_dir': os.path.join(VDB_DIR, 'nist'),
        'header'     : '### ',          # split on "### " headers
        'collection' : 'nist_ai_rmf',
    },
    {
        'name'       : 'Korean AI Basic Act',
        'key'        : 'kr',
        'file'       : 'kr_aibasicact_md.md',
        'persist_dir': os.path.join(VDB_DIR, 'kr_aibasicact'),
        'header'     : '### Article',   # split on "### Article" headers
        'collection' : 'kr_aibasicact',
    },
    {
        'name'       : 'EU AI Act',
        'key'        : 'eu',
        'file'       : 'eu_aiact_articles.md',
        'persist_dir': os.path.join(VDB_DIR, 'eu_aiact'),
        'header'     : '### Article',   # split on "### Article" headers
        'collection' : 'eu_aiact',
    },
]

print("[INFO] Regulation configuration loaded")
for reg in REGULATIONS:
    fpath = os.path.join(REG_DIR, reg['file'])
    size  = os.path.getsize(fpath) // 1024
    print(f"  {reg['name']:25s} | {reg['file']:35s} | {size:4d} KB")


# %%
# =============================================================================
# Cell 3. MD Loading and Chunking Function
# =============================================================================
def load_and_chunk(reg: dict) -> list[Document]:
    """
    Load a regulation MD file and split into chunks
    based on the specified header level.
    Each chunk is wrapped as a LangChain Document with metadata.
    """
    fpath = os.path.join(REG_DIR, reg['file'])
    with open(fpath, 'r', encoding='utf-8') as f:
        text = f.read()

    # Split on the regulation-specific header
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[(reg['header'].strip(), 'section')],
        strip_headers=False,
    )
    raw_chunks = splitter.split_text(text)

    # Wrap as Document with metadata
    docs = []
    for i, chunk in enumerate(raw_chunks):
        content = chunk.page_content.strip()
        if len(content) < 50:          # skip near-empty chunks
            continue
        docs.append(Document(
            page_content=content,
            metadata={
                'regulation' : reg['name'],
                'reg_key'    : reg['key'],
                'chunk_id'   : i,
                'source'     : reg['file'],
                'char_count' : len(content),
            }
        ))
    return docs


# Test chunking
print("[INFO] Chunking test")
for reg in REGULATIONS:
    docs = load_and_chunk(reg)
    chars = [len(d.page_content) for d in docs]
    print(f"  {reg['name']:25s} | chunks: {len(docs):3d} "
          f"| avg: {sum(chars)//len(chars):5d} chars "
          f"| min: {min(chars):4d} | max: {max(chars):5d}")


# %%
# =============================================================================
# Cell 4. Build ChromaDB Vector Stores
# =============================================================================
vector_stores = {}
chunk_stats   = []

for reg in REGULATIONS:
    print(f"\n[BUILD] {reg['name']} ...")

    docs = load_and_chunk(reg)

    # Build Chroma vector store (persisted to disk)
    vs = Chroma.from_documents(
        documents        = docs,
        embedding        = embeddings,
        collection_name  = reg['collection'],
        persist_directory= reg['persist_dir'],
    )

    vector_stores[reg['key']] = vs

    # Collect stats
    chars = [len(d.page_content) for d in docs]
    chunk_stats.append({
        'Regulation'     : reg['name'],
        'MD File'        : reg['file'],
        'Chunks'         : len(docs),
        'Avg Chars'      : round(sum(chars) / len(chars)),
        'Min Chars'      : min(chars),
        'Max Chars'      : max(chars),
        'Embed Model'    : EMBED_MODEL,
        'Persist Dir'    : reg['persist_dir'],
    })

    print(f"  [OK] {len(docs)} chunks embedded → {reg['persist_dir']}")

print("\n[INFO] All vector stores built and persisted.")


# %%
# =============================================================================
# Cell 5. Retrieval Test — Verify Search Quality
# =============================================================================
TEST_QUERIES = [
    {
        'query'    : 'What are the requirements for human oversight of high-risk AI credit scoring systems?',
        'reg_key'  : 'eu',
        'expected' : 'Article 14',
    },
    {
        'query'    : 'What obligations apply to AI business operators providing High-Impact AI for credit screening?',
        'reg_key'  : 'kr',
        'expected' : 'Article 34',
    },
    {
        'query'    : 'How does the GOVERN function address legal and regulatory requirements for AI systems?',
        'reg_key'  : 'nist',
        'expected' : 'GOVERN 1.1',
    },
]

print("[INFO] Retrieval Test (top-3 chunks per query)\n")
print("=" * 70)

for t in TEST_QUERIES:
    vs     = vector_stores[t['reg_key']]
    results = vs.similarity_search(t['query'], k=3)

    print(f"Query    : {t['query']}")
    print(f"Expected : {t['expected']}")
    print(f"Results  :")
    for i, doc in enumerate(results, 1):
        # Show first 120 chars of each retrieved chunk
        preview = doc.page_content[:120].replace('\n', ' ')
        print(f"  [{i}] {preview}...")
    print("-" * 70)


# %%
# =============================================================================
# Cell 6. RAG Retrieval Function (for Notebook 03)
# =============================================================================
# Regulation key → vector store mapping
REG_KEY_MAP = {
    'NIST_AI_RMF'      : 'nist',
    'KR_AI_BASIC_ACT'  : 'kr',
    'EU_AI_ACT'        : 'eu',
}

def retrieve_context(question: str, regulation: str, k: int = 3) -> str:
    """
    Retrieve top-k relevant chunks for a given question and regulation.

    Args:
        question   : The question string from the dataset
        regulation : Regulation key (e.g., 'NIST_AI_RMF')
        k          : Number of chunks to retrieve (default: 3)

    Returns:
        Concatenated context string from top-k chunks
    """
    reg_key = REG_KEY_MAP.get(regulation)
    if reg_key not in vector_stores:
        raise ValueError(f"[ERROR] Unknown regulation key: {regulation}")

    docs    = vector_stores[reg_key].similarity_search(question, k=k)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    return context


# Function test
print("[INFO] retrieve_context() function test")
sample_q   = "What are the penalties for non-compliance with High-Risk AI obligations?"
sample_reg = "EU_AI_ACT"
ctx = retrieve_context(sample_q, sample_reg, k=2)
print(f"  Question  : {sample_q}")
print(f"  Regulation: {sample_reg}")
print(f"  Context preview (first 300 chars):\n  {ctx[:300].replace(chr(10), ' ')}...")


# %%
# =============================================================================
# Cell 7. Table 3 — RAG Configuration Summary (for Paper)
# =============================================================================
table3 = pd.DataFrame(chunk_stats)

# Add shared configuration columns
table3['Chunking Strategy'] = 'MarkdownHeaderTextSplitter'
table3['Top-k Retrieval']   = 3
table3['Vector Store']      = 'ChromaDB (persistent)'

out_tbl = os.path.join(TABLE_DIR, 'table3_rag_config.csv')
table3.to_csv(out_tbl, index=False, encoding='utf-8-sig')

print("[Table 3] RAG Configuration Summary")
print(table3[['Regulation', 'Chunks', 'Avg Chars', 'Min Chars',
              'Max Chars', 'Chunking Strategy', 'Top-k Retrieval']].to_string(index=False))
print(f"\n[SAVE] {out_tbl}")
print(f"\n✅ Notebook 02 complete — Next: 03_llm_inference.ipynb")

[INFO] Environment setup complete
  Embedding model : text-embedding-3-small
  REG_DIR         : C:\Users\User\Downloads\학술\9_ai거버넌스_챗봇_ssci\financial_ai_governance\data\regulations
  VDB_DIR         : C:\Users\User\Downloads\학술\9_ai거버넌스_챗봇_ssci\financial_ai_governance\vectordb
[INFO] Regulation configuration loaded
  NIST AI RMF               | nist_rmf_md.md                      |   35 KB
  Korean AI Basic Act       | kr_aibasicact_md.md                 |   27 KB
  EU AI Act                 | eu_aiact_articles.md                |   52 KB
[INFO] Chunking test
  NIST AI RMF               | chunks:  11 | avg:  3255 chars | min:  256 | max:  7264
  Korean AI Basic Act       | chunks:  17 | avg:  1643 chars | min:  322 | max:  4248
  EU AI Act                 | chunks:  23 | avg:  2352 chars | min:  331 | max:  4525

[BUILD] NIST AI RMF ...
  [OK] 11 chunks embedded → ../vectordb\nist

[BUILD] Korean AI Basic Act ...
  [OK] 17 chunks embedded → ../vectordb\kr_aibasicact

[BUILD] EU AI Act